# Project NOVA: Cross-Execution Consensus

This notebook is the remote control for the `nova/` package. It holds no model logic. Run it top to bottom in a fresh Colab runtime.

1. Setup and environment check
2. Dependency audit and split audit (both must pass before any result counts)
3. Test suite
4. CPU smoke training of every model
5. Reduced end-to-end experiment (about 90 GPU minutes)
6. Results: GEI, learning curves, calibration, adaptive compute, attacks
7. Backend reliability (Experiment F)
8. Full preregistered experiments (off by default)
9. Research Console

Checkpoints and the ledger are written to Google Drive, so a disconnected runtime resumes where it stopped.

## 1. Setup and environment check

In [ ]:
import json
import os
import sys
import time

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = True  # keep checkpoints on Google Drive so they survive a disconnect
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
default_project = "/content/drive/MyDrive/NeuronX-Vaibhav-PIP" if IN_COLAB else os.getcwd()
PROJECT = os.environ.get("NOVA_PROJECT", default_project)
if not os.path.exists(os.path.join(PROJECT, "nova", "operator.py")):
    import glob
    hits = glob.glob("/content/drive/MyDrive/**/nova/operator.py", recursive=True) if IN_COLAB else []
    if not hits:
        raise FileNotFoundError(f"nova/ not found in {PROJECT}: upload the nova folder into that folder")
    PROJECT = os.path.dirname(os.path.dirname(hits[0]))
OUT = os.path.join(PROJECT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, PROJECT)
print("project:", PROJECT)
print("outputs:", OUT)

In [ ]:
import torch
import nova
from nova.train import amp_dtype, environment, pick_device

DEVICE = pick_device()
print(json.dumps(environment(), indent=2))
print("device:", DEVICE, "| mixed precision:", amp_dtype("auto", DEVICE))
print("code hash:", nova.code_hash())

## 2. Dependency audit and split audit

In [ ]:
from nova.audit import run_audit

report = run_audit(os.path.join(PROJECT, "nova"), os.path.join(PROJECT, "NOVA_Research.ipynb"))
print("dependency audit passed:", report["passed"])

In [ ]:
from nova.splits import Splits, split_audit

LEDGER = os.path.join(OUT, "ledger", "experiments.jsonl")
SPLITS = {}
for d in ("sequence", "grid", "graph"):
    SPLITS[d] = Splits(d, eval_tasks=200, ledger_path=LEDGER)
    rep = split_audit(SPLITS[d], samples=2000, report_path=os.path.join(OUT, f"split_audit_{d}.json"))
    print(f"{d:9s} audit passed | programs per split: {rep['manifest']['counts']}")

## 3. Test suite

In [ ]:
import unittest

suite = unittest.defaultTestLoader.discover(os.path.join(PROJECT, "nova", "tests"), top_level_dir=PROJECT)
result = unittest.TextTestRunner(verbosity=1).run(suite)
print("run:", result.testsRun, "| failures:", len(result.failures), "| errors:", len(result.errors), "| skipped:", len(result.skipped))
assert result.wasSuccessful(), "fix failing tests before running any experiment"

## 4. CPU smoke training (every model, a few steps)

In [ ]:
from nova import config as C
from nova.train import train_run

MODELS = ("nova", "ablated", "transformer", "transformer_cm", "recurrent", "heuristic")
smoke = []
for name in MODELS:
    rec = train_run(C.smoke_config(name), out_dir=os.path.join(OUT, "smoke"), experiment="smoke", device="cpu")
    smoke.append({"model": name, "end_state": rec["end_state"], "params": rec["params"]})
print(smoke)

## 5. Reduced end-to-end experiment

One seed, 600 updates, every model on the three static domains. Each run prints its progress every 100 steps. If the runtime disconnects, re-run the notebook: every finished run is skipped and an unfinished one resumes from its last checkpoint.

In [ ]:
RUN_REDUCED = True  # set False to skip straight to the console
DOMAINS = ("sequence", "grid", "graph")
RECORDS = []
if RUN_REDUCED:
    start = time.time()
    for d in DOMAINS:
        for name in MODELS:
            cfg = C.reduced_config(name, d)
            RECORDS.append(train_run(cfg, out_dir=os.path.join(OUT, "reduced"), experiment="reduced", device=DEVICE))
            print(f"{d:9s} {name:15s} {RECORDS[-1]['end_state']:10s} {(time.time() - start) / 60:6.1f} min")

## 6. Results

One seed only, so these are pipeline checks, not headline results. Headline claims need the full experiments in section 8.

In [ ]:
from IPython.display import HTML, display
from nova import console as CON
from nova import evaluate as E

budgets = E.budgets_from_records(RECORDS)
curves = E.curves_from_records(RECORDS)
heur = {d: curves.get("heuristic", {}).get(d, [0.0])[-1] for d in DOMAINS}
learned = {m: c for m, c in curves.items() if m != "heuristic"}
gei = E.gei_from_curves(learned, budgets, heur) if "transformer" in learned else {}
rows = [{"model": m, "GEI": v["gei"], **{f"score_{d}": s for d, s in v["domain_scores"].items()}} for m, v in gei.items()]
display(HTML(CON.html_table(rows, "GEI from the reduced run (1 seed: not a headline result)")))
for d in DOMAINS:
    display(HTML(CON.svg_learning_curves({m: c[d] for m, c in curves.items() if d in c}, budgets, f"{d}: mean OOD accuracy")))

In [ ]:
from nova.interface import ResearchBackend

BACKEND = ResearchBackend()
for name in ("nova", "transformer", "recurrent", "ablated"):
    ck = os.path.join(OUT, "reduced", "checkpoints", f"{name}_sequence_s0", "last.pt")
    status = BACKEND.load_checkpoint(ck)["status"] if os.path.exists(ck) else "missing: run section 5 first"
    print(f"{name:12s} {status}")
if "nova" in BACKEND.models:
    held_out = SPLITS["sequence"].eval_set("comp_ood", 100)
    res = E.evaluate_tasks(BACKEND.models["nova"], held_out, device="cpu")
    print("calibration:", E.calibration_report(res))
    print("adaptive compute:", E.adaptive_compute_report(res))
    print("attacks:", E.attack_suite(BACKEND.models["nova"], held_out[:50]))

## 7. Backend reliability (Experiment F)

In [ ]:
from nova.interface import reliability_suite

if "nova" in BACKEND.models:
    pool = [t for d in DOMAINS for t in SPLITS[d].eval_set("iid_val", 100)]
    rel = reliability_suite(BACKEND, pool, n_valid=1000, n_invalid=1000)
    print({k: v for k, v in rel.items() if k != "trace_sample"})
    print("20 traces to inspect (fixed rule, smallest sha256):", rel["trace_sample"])

## 8. Full preregistered experiments (off by default)

Experiments A, B and D follow the preregistration. They need roughly 26-49 GPU hours, so run them in pieces: every run resumes from its checkpoint and every outcome, including failures, goes into the ledger. The sealed family-OOD split is opened exactly once, at the very end, with `FINAL_EVAL = True`.

In [ ]:
RUN_FULL = False  # switch on only for the real experiments
FINAL_EVAL = False  # opens the sealed family-OOD split; do this once, after everything else is frozen
FULL_OUT = os.path.join(OUT, "full")
FULL = []
if RUN_FULL:  # Experiment A: 5 learned models + heuristic, 3 domains, 3 seeds
    for seed in (0, 1, 2):
        for d in DOMAINS:
            for name in MODELS:
                cfg = C.replace(C.RunConfig(), data={"domain": d}, train={"model_name": name, "seed": seed})
                FULL.append(train_run(cfg, out_dir=FULL_OUT, experiment="A", device=DEVICE))
if RUN_FULL:  # Experiment B: each mechanism removed on its own, plus the controls
    VARIANTS = ("nova_consensus_only", "nova_refine_only", "nova_no_persistent", "nova_no_deep_sup",
                "nova_no_aux", "nova_shuffled_r", "ablated_more_params")
    for seed, d in ((0, "sequence"), (1, "sequence"), (2, "sequence"), (0, "graph")):
        for name in VARIANTS:
            cfg = C.replace(C.RunConfig(), data={"domain": d}, train={"model_name": name, "seed": seed})
            FULL.append(train_run(cfg, out_dir=FULL_OUT, experiment="B", device=DEVICE))
if RUN_FULL:  # Experiment D: continual stream conditions
    CONDITIONS = [("nova", "frozen", "none"), ("nova", "none", "none"), ("nova", "none", "reservoir"),
                  ("nova", "rgp", "priority"), ("nova", "rgp", "none"), ("nova", "lowlr", "reservoir"),
                  ("transformer", "none", "reservoir"), ("transformer", "loss", "priority"),
                  ("recurrent", "none", "reservoir"), ("recurrent", "loss", "priority")]
    for seed in (0, 1, 2):
        for name, gate, replay in CONDITIONS:
            cfg = C.replace(C.RunConfig(), data={"domain": "stream"},
                            train={"model_name": name, "seed": seed, "gate": gate, "replay": replay,
                                   "updates": 3000, "budgets": (375, 750, 1500, 3000)})
            FULL.append(train_run(cfg, out_dir=FULL_OUT, experiment="D", device=DEVICE))
if RUN_FULL:  # Experiments C and E on the final sequence checkpoints
    for name in ("nova", "transformer", "transformer_cm", "recurrent", "ablated"):
        ck = os.path.join(FULL_OUT, "checkpoints", f"{name}_sequence_s0", "last.pt")
        if BACKEND.load_checkpoint(ck)["status"] == "ok":
            m = BACKEND.models[name]
            print(name, "surface:", E.extrapolation_surface(m, "sequence", n=50))
            print(name, "attacks:", E.attack_suite(m, SPLITS["sequence"].eval_set("comp_ood", 200)))
if RUN_FULL and FINAL_EVAL:  # the one and only opening of the sealed split
    for d in DOMAINS:
        sealed = SPLITS[d].final_eval_set(reason="final evaluation after all decisions were frozen")
        for name in BACKEND.models:
            print(d, name, E.evaluate_tasks(BACKEND.models[name], sealed, device="cpu")["task_acc"])

## 9. Research Console

Edit the request, press Validate, then Run. Outside Colab, use the callable cell below instead of the buttons.

In [ ]:
console = CON.launch(BACKEND)

In [ ]:
console.run(json.dumps(CON.EXAMPLE_REQUEST))
print("run record written to:", console.export(os.path.join(OUT, "console_run_record.json")))

In [ ]:
results = {"code_hash": nova.code_hash(), "environment": environment(), "smoke": smoke, "reduced": RECORDS,
           "gei_reduced": {m: {"gei": v["gei"], "domain_scores": v["domain_scores"]} for m, v in gei.items()}}
with open(os.path.join(OUT, "results_session.json"), "w") as fh:
    json.dump(results, fh, indent=2, default=str)
print("results:", os.path.join(OUT, "results_session.json"))
print("ledger:", os.path.join(OUT, "reduced", "ledger", "experiments.jsonl"))

## 10. Generate the remaining deliverables

Writes `model_card.md`, `test_report.md`, `failure_report.md` and `paper.md` into the project folder. Facts that already exist (test results, split manifests, the ledger) are filled in automatically. Anything that needs the full experiments in section 8, or your own judgement, is left as an explicit `PENDING` marker rather than invented, since the brief gives no credit for a conclusion written before the experiment that is supposed to produce it. Run this once now, and again after section 8 finishes.

In [ ]:
# Section 10. Generate the four remaining deliverables: model_card.md, test_report.md,
# failure_report.md, paper.md. Facts that already exist (test results, split manifests, the
# ledger) are filled in automatically. Anything that needs the full experiments (section 8) or
# your own judgement is left as an explicit PENDING marker, never invented: the brief gives no
# credit for a conclusion written before the experiment that is supposed to produce it.
import datetime
import os
import json
from nova import tasks as T

PENDING = "**PENDING** — fill in once the matching experiment has finished and you have judged the result."


def _get(name, default=None):
    return globals().get(name, default)


def _jsonl(path):
    rows = []
    if not os.path.exists(path):
        return rows
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:  # a crash mid-write can leave a broken last line
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def _latest(rows, key_fields):  # keep only the newest row per key (a resumed run writes several lines)
    best = {}
    for r in rows:
        key = tuple(r.get(f) for f in key_fields)
        if key not in best or r.get("end_time", 0) > best[key].get("end_time", 0):
            best[key] = r
    return list(best.values())


def _bytes(n):
    return f"{n / 2 ** 30:.2f} GiB" if n and n > 2 ** 20 else f"{n or 0} B"


FULL_LEDGER = _jsonl(os.path.join(OUT, "full", "ledger", "experiments.jsonl"))
RED_LEDGER = _latest(RECORDS, ("model", "domain", "seed")) if _get("RECORDS") else []
ALL_RUNS = RED_LEDGER + FULL_LEDGER
RESULT = _get("result")
REL = _get("rel")
GEI = _get("gei", {})  # the reduced-run GEI table from section 6 (never a headline result on its own)
FULL_RECS = _latest(FULL_LEDGER, ("model", "domain", "seed")) if FULL_LEDGER else []
GEI_FULL = {}  # headline GEI, computed below only if the full experiment actually produced curves
if FULL_RECS:
    full_curves = E.curves_from_records(FULL_RECS)
    full_budgets = E.budgets_from_records(FULL_RECS)
    full_heur = {d: full_curves.get("heuristic", {}).get(d, [0.0])[-1] for d in DOMAINS}
    learned_full = {m: c for m, c in full_curves.items() if m != "heuristic"}
    if "transformer" in learned_full:
        GEI_FULL = E.gei_from_curves(learned_full, full_budgets, full_heur)

print(f"facts available: {len(ALL_RUNS)} training runs, tests={'yes' if RESULT else 'no'}, "
      f"reliability={'yes' if REL else 'no'}, full experiment={'yes' if FULL_RECS else 'no (still PENDING)'}")


def write_model_card():
    rows = "\n".join(
        f"| {d} | {T.GENERATOR_VERSION} | {sp.manifest()['counts']['train']} | "
        f"{T.SEALED_FAMILIES.get(d, '-')} | {sp.manifest()['manifest_hash'][:16]}… |"
        for d, sp in SPLITS.items())
    comp_rows = "\n".join(
        f"| {r['model']} | {r['domain']} | {r.get('params', '?')} | {_bytes(r.get('peak_mem_bytes'))} | "
        f"{r.get('environment', {}).get('gpu', 'CPU')} |"
        for r in sorted(ALL_RUNS, key=lambda r: (r['model'], r['domain'])))
    text = f"""# NOVA Model Card

Code fingerprint: {nova.code_hash()}
Generated: {datetime.datetime.utcnow().isoformat()}Z
Environment this card describes: {json.dumps(environment())}

## Intended scope
NOVA is a research prototype built to test one question: can a learned operator called Cross-Execution
Consensus acquire and compose skills more efficiently than parameter- and compute-matched baselines, under
distribution shift, on the procedurally generated puzzles described in the preregistration? It is evaluated
only on the Procedural Generalization Suite defined in `nova/tasks.py`: symbolic sequences, relational grids,
graph programs, and a continual stream built from those three.

## Prohibited interpretations
- NOVA is not a general-intelligence system and makes no claim to be one.
- A result on these procedural puzzles must not be extrapolated to natural language, vision, or any task
  outside the Procedural Generalization Suite.
- A result from one seed or one domain is not a general claim; the decision rule in the preregistration
  requires at least two domains and all three seeds to agree before any claim of superiority is made.

## Data generators
| Domain | Generator version | Training programs | Sealed family (family-OOD) | Split manifest hash |
|---|---|---|---|---|
{rows}

No ARC private tasks are used, and no public ARC task is reproduced verbatim (see `nova/tasks.py`, section 6.1
of the technical design).

## Compute
| Model | Domain | Parameters | Peak memory observed | Device |
|---|---|---|---|---|
{comp_rows if comp_rows else '| (no runs recorded yet in this session) | | | | |'}

GPU budget for the full preregistered experiment: {PENDING} (measure from section 8's progress lines once it runs;
the technical design's estimate was 26-49 GPU hours after the declared reduction ladder, not yet confirmed).

## Known failure modes (declared in the preregistration before any result was seen)
- K = 2 demonstrations: a corrupted demonstration cannot be identified, because the cross-check matrix carries
  symmetric information with only two entries.
- Rules that no single demonstration pins down on its own.
- Very long inputs that need a global operation such as a full sort.
- Count outputs (`reduce(count)`) never seen as a value during training.
- Duplicated demonstrations gain extra weight; NOVA is not invariant to duplicates (see Experiment E).

## Reproducibility envelope
- Same-seed, same-environment replay is claimed as bitwise exact (`nova/tests/test_reproducibility.py`,
  `nova/tests/test_integration.py::test_resume_reproduces_next_step_exactly`).
- Cross-platform results (a different GPU, a different PyTorch release) are claimed only as numerically close,
  per PyTorch's own reproducibility documentation.
- Test suite status at the time this card was generated: {"ALL PASSED" if RESULT and RESULT.wasSuccessful() else PENDING}.
"""
    path = os.path.join(PROJECT, "model_card.md")
    open(path, "w").write(text)
    return path


def write_test_report():
    test_dir = os.path.join(PROJECT, "nova", "tests")
    files = sorted(f for f in os.listdir(test_dir) if f.startswith("test_"))
    if RESULT:
        status = ("ALL PASSED" if RESULT.wasSuccessful() else
                   f"{len(RESULT.failures)} failing, {len(RESULT.errors)} erroring (see below)")
        fail_rows = "\n".join(f"- FAIL: `{t}`" for t, _ in RESULT.failures) or "- none"
        err_rows = "\n".join(f"- ERROR: `{t}`" for t, _ in RESULT.errors) or "- none"
        skip_rows = "\n".join(f"- SKIP ({reason}): `{t}`" for t, reason in RESULT.skipped) or "- none"
        counts = f"{RESULT.testsRun} tests run, {len(RESULT.failures)} failed, {len(RESULT.errors)} errored, {len(RESULT.skipped)} skipped"
    else:
        status, fail_rows, err_rows, skip_rows, counts = PENDING, "-", "-", "-", "re-run section 3 to populate this"
    peak = max((r.get("peak_mem_bytes", 0) or 0 for r in ALL_RUNS), default=0)
    text = f"""# NOVA Test Report

Code fingerprint: {nova.code_hash()}
Generated: {datetime.datetime.utcnow().isoformat()}Z

## Test inventory
{len(files)} files under `nova/tests/`: {', '.join(files)}.
`tools/seed_bugs.py` additionally plants each of the brief's eight seeded-bug types and checks that at least
one test fails for each; run it separately (`python tools/seed_bugs.py`), since it takes 5-10 minutes.

## Pass/fail status
{counts}
Overall: **{status}**

Failures:
{fail_rows}

Errors:
{err_rows}

Skipped (each with its reason, since a skip is not a failure):
{skip_rows}

## Coverage map by requirement (condensed; see Part III of the technical documentation for the full table)
| Brief requirement | Test file |
|---|---|
| Operator shapes, invariants, edge cases, fail-fast on NaN | `test_operator.py` |
| Every model's shapes, batch equivalence, parameter matching | `test_shapes.py` |
| gradcheck, finite differences, gradient flow, AMP agreement | `test_gradients.py` |
| Mask correctness, no information leaks (query, future tokens, hidden fields) | `test_masks.py` |
| Split audit passes and catches planted leaks; sealed split discipline | `test_splits.py` |
| Checkpoint round trip, corruption, portability, RNG restore after GPU load | `test_checkpoint.py` |
| Fresh import, exact resume, backend contracts, console, reliability suite | `test_integration.py` |
| Same-seed replay, label-shuffle collapse, support-set ablation, dependency audit | `test_reproducibility.py` |
| Latency, profiler, complexity bound, OOM recovery, adaptive-compute savings | `test_performance.py` |

## Gradient-check output
`test_gradients.py::test_gradcheck_operator_float64` runs PyTorch's `gradcheck` on the operator in double
precision. `gradgradcheck` (second-order gradients) is explicitly **not claimed** for CEC; the test file
documents this as a declared decision rather than an oversight.

## Reproducibility checks
- Same-seed replay: `test_reproducibility.py::test_same_seed_replay_is_exact`.
- Bitwise exact resume (two steps plus every weight): `test_integration.py::test_resume_reproduces_next_step_exactly`.
- RNG restore after a checkpoint is loaded onto a GPU: `test_checkpoint.py::test_rng_restore_after_loading_onto_gpu`
  (this test only runs where a GPU is present; it is a regression test for a real bug found on the first Colab
  GPU run, see the failure report).

## Backend reliability (Experiment F)
{json.dumps({k: v for k, v in REL.items() if k != 'trace_sample'}, indent=2) if REL else PENDING + " (run section 7)"}

## Performance profile
Peak memory observed across every run in this session: {_bytes(peak)}.
Declared cap from the brief: 12 GiB per run. Full profiler traces are printed by
`test_performance.py::test_profiler_names_top_operations`; paste that cell's output here for the submission.
"""
    path = os.path.join(PROJECT, "test_report.md")
    open(path, "w").write(text)
    return path


def write_failure_report():
    known = [  # real failures found and fixed while building this system, in the order they were found
        ("Behavioural split leak from commuting rules",
         "`sort` then `rotate` was held out, but `map_affine` then `dedup` and `dedup` then `map_affine` commute, "
         "so one order was accidentally left in training.",
         "Added a behaviour-fingerprint filter (`nova/splits.py::_behaviour`) that drops any held-out program "
         "whose output matches a training program on fixed probes, not just a name-based predicate."),
        ("The brief's own suggested resume check misses a real bug",
         "Comparing only the next step's loss after resume did not change when the optimizer's state (momentum) "
         "was silently dropped, because that loss is computed before the optimizer update is applied.",
         "Strengthened `test_resume_reproduces_next_step_exactly` to compare two steps and every model weight."),
        ("Transformer-CM cannot match NOVA's compute on grid and graph domains",
         "A single Transformer layer pass already costs 1.8-2.3x NOVA's FLOPs at the grid/graph test sizes used, "
         "so adding loops only increases the gap; `match_compute` now warns and records the achieved ratio.",
         "Report the actual ratio next to every Transformer-CM result rather than assuming it is matched."),
        ("RNG restore crashed after a checkpoint was loaded onto a GPU",
         "`torch.set_rng_state` requires a CPU byte tensor; `map_location='cuda'` moved the saved CPU generator "
         "state onto the GPU, which then failed with `TypeError: RNG state must be a torch.ByteTensor`.",
         "Fixed `restore_rng` to move the stored RNG tensors back to the CPU before restoring; added "
         "`test_rng_restore_after_loading_onto_gpu` as a regression test."),
        ("A test compared GPU weights against a CPU copy directly",
         "`test_idempotent_and_no_training_during_inference` raised a device-mismatch error on a GPU runtime, "
         "which looked like a model bug but was a test bug: the backend may legitimately run on the GPU.",
         "Fixed the test to compare both tensors on the CPU."),
        ("Grid and graph puzzles were too large for the 90-minute reduced-run budget",
         "Transformer context length reached up to ~2,081 tokens on the larger grid test set, making the "
         "reduced experiment take many hours instead of 90 minutes.",
         "Shrank the grid and graph size regimes, capped every model's answer length to the demonstrations' own "
         "length, and added a key/value cache to the Transformer baseline so decoding reuses earlier work."),
    ]
    for r in ALL_RUNS:
        if r.get("end_state") in ("failed", "diverged", "cancelled"):
            known.append((f"Run {r['model']}/{r['domain']} seed {r.get('seed')} ended as {r['end_state']}",
                          r.get("failure_reason", "no reason recorded"),
                          "Diagnose using the ledger's `events` field and this run's checkpoint, then rerun."))
    rows = "\n".join(f"| {i+1} | {t} | {e} | {f} |" for i, (t, e, f) in enumerate(known))
    extra = max(0, 10 - len(known))  # how many more the brief's minimum of ten still needs
    tail = (f"\n\n{extra} more entries are required to reach the brief's minimum of ten. These should come from "
            f"real negative results in Experiments B, D and E once they run (for example: an ablation that shows "
            f"no measurable gain, an attack NOVA fails outright, or a domain where GEI ends up below 1.0) rather "
            f"than being invented now. {PENDING}") if extra else ""  # honest note instead of padding
    text = f"""# NOVA Failure Report

Generated: {datetime.datetime.utcnow().isoformat()}Z. {len(known)} failures recorded so far.

| # | Failure | Evidence | Proposed follow-up |
|---|---|---|---|
{rows}{tail}
"""
    path = os.path.join(PROJECT, "failure_report.md")
    open(path, "w").write(text)
    return path


def write_paper():
    def gei_table(table):
        if not table:
            return PENDING + " (needs section 8 with RUN_FULL = True)"
        rows = "\n".join(f"| {m} | {v['gei']:.3f} | " + ", ".join(f"{d}={s:.3f}" for d, s in v["domain_scores"].items()) + " |"
                         for m, v in table.items())
        return f"| Model | GEI | Domain scores |\n|---|---|---|\n{rows}"
    claims = [  # the preregistration's predictions, mapped to their decisive test
        ("C1: higher skill-acquisition efficiency under shift", "cross-execution verification before reuse",
         "GEI above both the Transformer and Transformer-CM on >=2 domains, all 3 seeds", "Experiment A"),
        ("C2: the gain is the operator, not size or compute", "consensus weighting and refinement",
         "beats Ablated NOVA, the more-parameters control, and the shuffled-R control", "Experiment B"),
        ("C3: robust to one corrupted demonstration (K>=3)", "reliability weights discount bad evidence",
         "smaller accuracy drop than the Transformer under attack 2", "Experiment E"),
        ("C4: invariant to demonstration order", "aggregation is over sets, not sequences",
         "identical outputs under permutation, verified not just claimed", "test_operator.py::test_I1, hidden eval"),
        ("C5: adaptive effort saves real compute", "residual-based halting with batch compaction",
         "fewer executor calls and less time on easy tasks at matched accuracy", "Experiment B, fixed-T control"),
        ("C6: better-calibrated confidence", "confidence follows the fit to held-out demonstrations",
         "lower ECE than temperature-scaled softmax on OOD splits", "calibration_report() in section 6"),
        ("C7: better length extrapolation", "recurrent executor with no absolute positions",
         "smaller accuracy gap beyond training length than the Transformer's", "Experiment C"),
        ("C8: residual-gated plasticity limits forgetting", "weight updates scaled by the fit residual",
         "less forgetting than ungated online updates at the same replay cap", "Experiment D"),
    ]
    claim_rows = "\n".join(f"| {c} | {m} | {p} | {t} | {PENDING if not GEI_FULL else 'see Results'} | {PENDING} | {PENDING} |"
                           for c, m, p, t in claims)
    text = f"""# Project NOVA: Cross-Execution Consensus

Status: {"full experiment results are included below" if GEI_FULL else "DRAFT — the full preregistered experiment (section 8) has not run yet; every numeric claim below is PENDING"}.
Generated: {datetime.datetime.utcnow().isoformat()}Z. Code fingerprint: {nova.code_hash()}.

## Abstract
We propose Cross-Execution Consensus (CEC), an operator that forms a rule from a few demonstrations, verifies
each candidate rule by running it on the other demonstrations, and refines the trusted combination with its own
execution residual until it explains the evidence or a step budget is reached. {"Across the preregistered seeds "
"and domains, NOVA's measured Generalization Efficiency Index (GEI) was: see the Results section." if GEI_FULL else PENDING}
We do not claim NOVA is, or leads to, general intelligence; every claim is confined to the procedural puzzles
measured here.

## Related work
Conditional Neural Processes mean-pool per-example encodings; CEC replaces the uniform mean with weights
derived from cross-execution error, and the mean-pooled setting is our degenerate ablation. The Latent Program
Network refines a latent program by test-time gradient search; CEC refines with a learned forward update driven
by execution residuals instead, so the brief's clause on test-time gradient search does not apply. Execution-
guided program synthesis, CodeT and AlphaCode verify discrete candidate programs by running them; CEC verifies
continuous latent rules with a learned, differentiable executor. Adaptive Computation Time and the Universal
Transformer learn a halting probability over every token's state; CEC halts a single rule vector on an
externally verifiable residual. Order invariance is inherited from Deep Sets and is not claimed as novel.

## Method
See `preregistration.md` for the exact mathematical definition of the primitive, and `nova/operator.py` for the
implementation. In short: K demonstrations each propose a rule; every proposed rule is executed on every other
demonstration to build a K x K error matrix; execution-verified trust and reliability weights combine the
guesses; the combined rule is refined by a learned update driven by its own execution residual; refinement
halts when the residual is small, stops improving, or a step budget is reached.

## Complexity
NOVA costs roughly 5-6.5x a parameter-matched Transformer's forward FLOPs per task, because of the K(K-1)
cross-execution passes (measured in `nova/baselines.py::measure_flops`). We make no asymptotic efficiency
claim; the only compute claim is the adaptive saving on easy tasks (Experiment B, fixed-T control), which is
why a compute-matched Transformer (Transformer-CM) is included as a baseline rather than only a size-matched one.

## Experimental design
Three procedural domains (symbolic sequences, relational grids, graph programs) plus a continual stream, each
with IID, compositional-OOD, structural-OOD and a sealed family-OOD split (`nova/splits.py`). Five learned
baselines plus a non-learning heuristic, every learned model matched to NOVA's parameter count within 5%
(`nova/baselines.py::match_parameters`). Primary metric: GEI, our reading of the brief's metric since its
equations and supplied evaluator were missing from the PDF we received (open question to the Induction Team).

## Results
Reduced-run pilot (600 steps, one seed; a pipeline check, not a headline result):
{gei_table(GEI)}

Full preregistered experiment (10,000 steps, 3 seeds, Experiment A):
{gei_table(GEI_FULL)}

## Ablations (Experiment B)
{PENDING if not FULL_RECS else "populate from FULL_RECS filtered to experiment == 'B'"}

## Claims table
| Claim | Causal mechanism | Prediction | Decisive test | Result | Confidence | Alternative explanation |
|---|---|---|---|---|---|---|
{claim_rows}

## Limitations
NOVA's compute cost (5-6.5x a same-size Transformer) means a compute-matched comparison is essential and is
included. Every claim is confined to the procedural task families defined in `nova/tasks.py`. The GEI formula
used here is our own reading of the brief's text, not the brief's own (missing) equations.

## Conclusion
{PENDING} — write this only after the decision rule in the preregistration has been applied to real Experiment
A and B results; do not state a conclusion before that.
"""
    path = os.path.join(PROJECT, "paper.md")
    open(path, "w").write(text)
    return path


for fn in (write_model_card, write_test_report, write_failure_report, write_paper):
    p = fn()
    pending_count = open(p).read().count("PENDING")
    print(f"wrote {p}  ({pending_count} section(s) marked PENDING)")